# 07 Failure Drills and Recovery Validation (LiteLLM, 2026)

## What This Lesson Is
Run explicit failure drills and assert expected fallback/recovery behavior.

## Scientific Lens
- Concept: Chaos-style reliability validation
- Measure: Drill pass rate and mean recovery time
- Validity Limit: Synthetic drills are useful only when continuously updated for real architecture changes.


## How It Works
1. Define scenario matrix with expected outcomes.
2. Automate pass/fail assertions.
3. Run a live degraded-path exercise.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Drill count:", 3)


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
scenarios = [
    {"name": "primary_timeout", "expected": "fallback_secondary", "actual": "fallback_secondary"},
    {"name": "secondary_auth_error", "expected": "fallback_local", "actual": "fallback_local"},
    {"name": "all_unavailable", "expected": "fail_fast", "actual": "fail_fast"},
]

results = []
for s in scenarios:
    passed = s["expected"] == s["actual"]
    results.append({"scenario": s["name"], "passed": passed})

print(results)
assert all(r["passed"] for r in results)


In [ ]:
# Live Demo
import os

try:
    from litellm import completion
except Exception as exc:
    print(f"Skipping live failure drill: litellm unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live failure drill: OPENAI_API_KEY not set.")
    else:
        # intentionally attempt invalid model first to force fallback behavior
        models = ["openai/not-a-real-model", "openai/gpt-4.1-mini"]
        for model in models:
            try:
                r = completion(
                    model=model,
                    messages=[{"role": "user", "content": "One sentence: what is a failure drill?"}],
                    api_key=api_key,
                    timeout=20,
                )
                print("served by:", model)
                print(r.choices[0].message.content.strip())
                break
            except Exception as exc:
                print(f"model failed ({model}): {exc}")


## Applied Labs
1. Add a new drill for partial degradation (high latency but no hard failure).
2. Capture recovery time in milliseconds for each scenario.
3. Fail one drill intentionally and validate CI gate behavior.

## Validation Checklist
- Every drill has explicit expected behavior.
- Pass/fail criteria are machine-checkable.
- Live drill demonstrates fallback from an intentionally broken path.

## Further Reading
- [Principles of Chaos Engineering](https://principlesofchaos.org/)
- [Netflix Chaos Engineering](https://netflixtechblog.com/tagged/chaos-engineering)
- [LiteLLM Reliability Features](https://docs.litellm.ai/docs/routing)
